In [ ]:
!pip install azure-storage-blob pandas numpy scikit-learn tensorflow

import os
import json
import pandas as pd
import numpy as np
from io import StringIO
from azure.storage.blob import BlobServiceClient
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tqdm.notebook import tqdm
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
AZURE_CONNECTION_STRING = ""


In [ ]:
# Full fixed cell: FAST ASYNC AZURE CACHE + NN TRAIN (works with AZURE_CONNECTION_STRING)
import os, asyncio, json, shutil, math, sys
import numpy as np, pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm
from azure.storage.blob.aio import BlobServiceClient
from azure.core.exceptions import ResourceNotFoundError, AzureError

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
!pip install -q nest_asyncio
import nest_asyncio
nest_asyncio.apply()

sns.set()

# ---------------- CONFIG ----------------
CSV_FILE = 'human_trueskill_rankings.csv'                # your CSV with rank_ai
CONTAINER_NAME = "london"                   # adjust if different
SEGMENTATION_FOLDER = "SemanticSegmentation/json"
DETECTION_FOLDER = "YoloV10/conf_0.5/labels"
LOCAL_CACHE_DIR = 'azure_cached_blobs'     # local storage for downloaded blobs
CACHE_FILE = 'feature_data_cache_nn.npz'   # aggregated feature cache
MODEL_OUT = 'safety_score_model_nn.keras'
TRAINING_CURVES = 'training_curves_nn.png'
CONFUSION_MATRIX = 'confusion_matrix_nn.png'

SIGMA_THRESHOLD = 4.0
NUM_TIERS = 3
RANDOM_STATE = 42

# Tunable concurrency — 32 works well in Colab, lower if you see throttling
MAX_CONCURRENCY = 32
DOWNLOAD_TIMEOUT_SECONDS = 60 * 2

# ---------------- FEATURE MAPPINGS ----------------
SEG_CLASSES_TO_KEEP = {1:0,2:1,11:2,12:3,19:4,24:5,26:6,29:7,32:8,42:9,48:10,51:11,54:12,56:13,58:14,62:15,64:16,66:17}
OBJ_CLASSES_TO_KEEP = {0:0,1:1,2:2,3:3,5:4,6:5,7:6,9:7,11:8,13:9,24:10}

SEG_NAMES = ['SEG_Bicycle','SEG_Bicyclist','SEG_Building','SEG_Bus','SEG_Car','SEG_Curb','SEG_Crosswalk','SEG_FireHydrant','SEG_LaneMarking','SEG_Parking','SEG_Road','SEG_Sidewalk','SEG_StreetLight','SEG_TrafficLight','SEG_TrafficSign','SEG_Truck','SEG_Unlabeled','SEG_Vegetation']
OBJ_NAMES = ['YOLO_Person','YOLO_Bicycle','YOLO_Car','YOLO_Motorcycle','YOLO_Bus','YOLO_Train','YOLO_Truck','YOLO_TrafficLight','YOLO_StopSign','YOLO_Bench','YOLO_Backpack']
RAW_FEATURE_NAMES = SEG_NAMES + OBJ_NAMES
RAW_FEATURE_MAP = {name:i for i,name in enumerate(RAW_FEATURE_NAMES)}
MAX_SEG_LABEL = len(SEG_CLASSES_TO_KEEP)
MAX_OBJ_LABEL = len(OBJ_CLASSES_TO_KEEP)

# ---------------- parsers ----------------
def parse_segmentation(json_data):
    features = np.zeros(MAX_SEG_LABEL, dtype=float)
    try:
        if isinstance(json_data, (dict,)):
            data = json_data
        else:
            data = json.loads(json_data)
        h = int(data.get('height', 1))
        w = int(data.get('width', 1))
        total_pixels = max(1, h * w)
        for item in data.get('label_histogram', []):
            original_idx = item.get('label_id')
            if original_idx in SEG_CLASSES_TO_KEEP:
                new_idx = SEG_CLASSES_TO_KEEP[original_idx]
                features[new_idx] = item.get('pixels', 0) / total_pixels
    except Exception:
        pass
    return features

def parse_yolo(yolo_data):
    features = np.zeros(MAX_OBJ_LABEL, dtype=float)
    try:
        if not isinstance(yolo_data, str):
            yolo_data = str(yolo_data)
        lines = yolo_data.strip().split('\n')
        for line in lines:
            parts = line.split()
            if len(parts) >= 1:
                try:
                    original_idx = int(parts[0])
                except Exception:
                    continue
                if original_idx in OBJ_CLASSES_TO_KEEP:
                    new_idx = OBJ_CLASSES_TO_KEEP[original_idx]
                    features[new_idx] += 1.0
    except Exception:
        pass
    return features

# ---------------- async download helpers ----------------
async def download_blobs_concurrently(blob_service_client, container_name, blob_names_to_download, local_dir, concurrency=32):
    os.makedirs(local_dir, exist_ok=True)
    sem = asyncio.Semaphore(concurrency)
    container_client = blob_service_client.get_container_client(container_name)

    async def _download_single(blob_name):
        local_path = os.path.join(local_dir, *blob_name.split('/'))
        local_dirpath = os.path.dirname(local_path)
        os.makedirs(local_dirpath, exist_ok=True)
        # If file already exists (non-zero size), skip
        if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
            return blob_name, True, "skipped"
        async with sem:
            try:
                blob_client = container_client.get_blob_client(blob_name)
                stream = await blob_client.download_blob()
                data = await stream.readall()
                # write binary bytes
                with open(local_path, 'wb') as f:
                    f.write(data)
                return blob_name, True, "downloaded"
            except ResourceNotFoundError:
                return blob_name, False, "not_found"
            except AzureError as e:
                return blob_name, False, f"azure_error:{e}"
            except Exception as e:
                return blob_name, False, f"error:{e}"

    results = []
    tasks = [asyncio.create_task(_download_single(bn)) for bn in blob_names_to_download]
    # iterate as completed with progress bar
    for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Download blobs"):
        res = await fut
        results.append(res)
    return results

# ---------------- main logic ----------------
async def build_cache_and_train_async():
    if not os.path.exists(CSV_FILE):
        raise FileNotFoundError(f"CSV not found at {CSV_FILE}")
    df = pd.read_csv(CSV_FILE)
    initial = len(df)
    df = df[df['human_sigma'] <= SIGMA_THRESHOLD].copy()
    print(f"Filtered human_sigma > {SIGMA_THRESHOLD}: {initial} -> {len(df)}")

    # create tiers from rank_human
    # qcut can drop duplicates; try to guard if it results in fewer bins
    try:
        df['safety_score'] = pd.qcut(df['rank_human'], q=NUM_TIERS, labels=False, duplicates='drop').astype(int) + 1
    except Exception:
        # fallback: rank into equal-width bins
        df['safety_score'] = pd.cut(df['rank_human'], bins=NUM_TIERS, labels=False, include_lowest=True).astype(int) + 1
    df['safety_tier_idx'] = df['safety_score'] - 1
    df['basename'] = df['image_name'].apply(lambda s: os.path.splitext(os.path.basename(str(s)))[0])
    df['root'] = df['basename'].apply(lambda s: s.split('_')[0])

    exact_map, root_map, data_map = defaultdict(list), defaultdict(list), {}
    for idx, row in df.iterrows():
        exact_map[row['basename']].append(idx)
        root_map[row['root']].append(idx)
        data_map[idx] = {'target': int(row['safety_tier_idx']), 'seg_features': np.zeros(MAX_SEG_LABEL, dtype=float), 'obj_features': np.zeros(MAX_OBJ_LABEL, dtype=float), 'has_seg': False, 'has_obj': False}


    blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
    container_client = blob_service_client.get_container_client(CONTAINER_NAME)

    print("Listing blobs in container (this can be a bit slow but is required to find matching files)...")
    # collect blobs with async comprehension then iterate with tqdm
    blobs = [b async for b in container_client.list_blobs()]
    blob_names_to_download = []
    blob_to_candidate_idxs = {}

    for blob in tqdm(blobs, desc="Scanning blobs"):
        name = blob.name
        filename = os.path.basename(name)
        base = os.path.splitext(filename)[0]
        root = base.split('_')[0]

        candidate_idxs = exact_map.get(base, [])
        if not candidate_idxs:
            candidate_idxs = root_map.get(root, [])
            # only accept single-root mapping to avoid ambiguous assignment
            if len(candidate_idxs) != 1:
                candidate_idxs = []

        if not candidate_idxs:
            continue

        # only download segmentation jsons or detection txts
        if (SEGMENTATION_FOLDER in name and name.endswith('.json')) or (DETECTION_FOLDER in name and name.endswith('.txt')):
            blob_names_to_download.append(name)
            blob_to_candidate_idxs[name] = candidate_idxs

    print("Blobs to download (matching seg/yolo & CSV rows):", len(blob_names_to_download))
    if len(blob_names_to_download) == 0:
        raise RuntimeError("No matching seg/yolo blobs found in container for CSV rows. Check SEGMENTATION_FOLDER/DETECTION_FOLDER names and blob layout.")

    # Run concurrent downloads into LOCAL_CACHE_DIR
    os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
    print(f"Starting parallel downloads with concurrency={MAX_CONCURRENCY} ...")
    results = await download_blobs_concurrently(blob_service_client, CONTAINER_NAME, blob_names_to_download, LOCAL_CACHE_DIR, concurrency=MAX_CONCURRENCY)

    # Summarize download results
    succ = sum(1 for r in results if r[1])
    fail = len(results) - succ
    print(f"Download results: success {succ}, failed {fail}")

    # Parse local files back into features similar to earlier local build
    X_raw_list = []
    y_list = []
    files_parsed = 0
    for blob_name, candidate_idxs in blob_to_candidate_idxs.items():
        local_path = os.path.join(LOCAL_CACHE_DIR, *blob_name.split('/'))
        if not os.path.exists(local_path):
            continue
        if local_path.endswith('.json'):
            # segmentation
            try:
                with open(local_path, 'r', encoding='utf-8') as f:
                    content = f.read()
            except Exception:
                continue
            segf = parse_segmentation(content)
            for idx in candidate_idxs:
                data_map[idx]['seg_features'] = segf
                data_map[idx]['has_seg'] = True
            files_parsed += 1

        elif local_path.endswith('.txt'):
            try:
                with open(local_path, 'r', encoding='utf-8') as f:
                    content = f.read()
            except Exception:
                continue
            objf = parse_yolo(content)
            for idx in candidate_idxs:
                data_map[idx]['obj_features'] = objf
                data_map[idx]['has_obj'] = True
            files_parsed += 1

    print("Parsed local files into data_map entries:", files_parsed)

    # Build X_raw and y from data_map entries that have seg or obj
    for v in data_map.values():
        if v['has_seg'] or v['has_obj']:
            segf_full = np.zeros(len(SEG_NAMES), dtype=float)
            objf_full = np.zeros(len(OBJ_NAMES), dtype=float)
            # place kept-class features into first parts of segf_full / objf_full
            for new_idx in range(len(v['seg_features'])):
                if new_idx < len(segf_full):
                    segf_full[new_idx] = v['seg_features'][new_idx]
            for new_idx in range(len(v['obj_features'])):
                if new_idx < len(objf_full):
                    objf_full[new_idx] = v['obj_features'][new_idx]
            raw_vec = np.concatenate((segf_full, objf_full))
            X_raw_list.append(raw_vec)
            y_list.append(v['target'])

    if len(X_raw_list) == 0:
        raise RuntimeError("No data rows had seg or obj after parsing downloaded blobs.")

    X_raw = np.array(X_raw_list)
    y = np.array(y_list)
    print("Built raw feature matrix:", X_raw.shape, "targets:", y.shape)

    # AGGREGATE into 8 features (same as before)
    def process_features_v2(X_raw, raw_feature_map):
      """
      Replaces the old 'aggregate_features'.
      1. Keeps all raw segmentation % and object counts.
      2. Adds 'Density' features (Count / Area) to give context.
      """
      # 1. Start with the raw features (Shape: N x 29)
      # We will append new features to this matrix.
      features_list = [X_raw]

      # Helper to get column index by name
      def get_col(name):
          return X_raw[:, raw_feature_map[name]]

      # --- Feature Engineering: Density & Interactions ---
      # Epsilon to prevent division by zero
      eps = 1e-6

      # Feature 30: Car Density (How many cars per unit of road?)
      # High traffic on small road = Dangerous. Low traffic on huge road = Safe.
      road_area = get_col('SEG_Road') + eps
      car_count = get_col('YOLO_Car') + get_col('YOLO_Truck') + get_col('YOLO_Bus')
      car_density = car_count / road_area
      features_list.append(car_density.reshape(-1, 1))

      # Feature 31: Pedestrian Density on Sidewalks
      # Lots of people on a tiny sidewalk = Dangerous/Spillover.
      sidewalk_area = get_col('SEG_Sidewalk') + eps
      ped_count = get_col('YOLO_Person')
      ped_density = ped_count / sidewalk_area
      features_list.append(ped_density.reshape(-1, 1))

      # Feature 32: Bicycle Interaction (Bikes vs Cars)
      # If there are bikes AND cars, it's more complex/dangerous than just bikes.
      bike_count = get_col('YOLO_Bicycle') + get_col('SEG_Bicyclist')
      bike_car_interaction = bike_count * car_count
      features_list.append(bike_car_interaction.reshape(-1, 1))

      # Feature 33: Obstacle Dominance
      # Ratio of obstacles (poles, hydrants) to walkable space
      obstacles = get_col('SEG_TrafficLight') + get_col('SEG_TrafficSign') + get_col('SEG_StreetLight') + get_col('SEG_FireHydrant')
      walkable = get_col('SEG_Sidewalk') + get_col('SEG_Crosswalk') + eps
      obstacle_ratio = obstacles / walkable
      features_list.append(obstacle_ratio.reshape(-1, 1))

      # Combine everything
      X_new = np.hstack(features_list)

      # Replace NaNs with 0 (just in case)
      X_new = np.nan_to_num(X_new)

      return X_new

    X_agg = process_features_v2(X_raw, RAW_FEATURE_MAP)
    print("Aggregated features shape:", X_agg.shape)

    # Save aggregated cache
    np.savez_compressed(CACHE_FILE, features=X_agg, targets=y)
    print("Saved aggregated cache to:", CACHE_FILE)
    return True

# ---------------- Execute async build (create cache) ----------------
# Make sure AZURE_CONNECTION_STRING is set in Colab environment
# ---------------- Load or build cache ----------------
if os.path.exists(CACHE_FILE):
    print("Found existing feature cache. Loading...")
else:
    print("No cache found. Building cache from Azure...")
    asyncio.run(build_cache_and_train_async())

print("Loading aggregated cache...")
d = np.load(CACHE_FILE, allow_pickle=True)
X_agg = d['features']
y = d['targets']
print("Loaded aggregated cache:", X_agg.shape, y.shape)


# ---------------- Load cache and train NN ----------------
d = np.load(CACHE_FILE, allow_pickle=True)
X_agg = d['features']
y = d['targets']
print("Loaded aggregated cache:", X_agg.shape, y.shape)

# ---------- Diagnostics, quick baselines, and improved NN training ----------

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.utils import class_weight
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
import matplotlib.pyplot as plt

# Basic diagnostics
print("Feature matrix shape:", X_agg.shape)
print("Label shape:", y.shape)
unique, counts = np.unique(y, return_counts=True)
print("Class distribution (label:count):", dict(zip(unique, counts)))
majority_acc = counts.max() / counts.sum()
print(f"Majority-class baseline accuracy = {majority_acc:.3f}")

# Optionally try a quick log(1+x) transform on features that are counts/positive:
X_log = np.log1p(X_agg)   # safe for positive features (works for small values too)

# Prepare splits (stratified)
X_temp, X_test, y_temp, y_test = train_test_split(X_log, y, test_size=0.10, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1111111, stratify=y_temp, random_state=RANDOM_STATE)
print("Train/Val/Test shapes:", X_train.shape, X_val.shape, X_test.shape)

# Scale (RobustScaler is good for outliers)
scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

# ---------- Quick classical baselines ----------
print("\nTraining quick baselines (LogisticRegression, RandomForest)...")

# Logistic Regression (multinomial)
lr = LogisticRegression(max_iter=500, multi_class='multinomial', solver='lbfgs', C=1.0)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
print("LogReg test acc:", accuracy_score(y_test, y_pred_lr))
print("LogReg classification report:")
print(classification_report(y_test, y_pred_lr, digits=4))

# Random Forest (gives feature importance)
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)   # RF can handle unscaled features
y_pred_rf = rf.predict(X_test)
print("RandomForest test acc:", accuracy_score(y_test, y_pred_rf))
print("RandomForest classification report:")
print(classification_report(y_test, y_pred_rf, digits=4))

# Plot RF feature importances
fi = rf.feature_importances_
plt.figure(figsize=(8,4))
plt.bar(range(len(fi)), fi)
plt.xticks(range(len(fi)), RAW_FEATURE_NAMES[:len(fi)], rotation=60, ha='right')
plt.title("RandomForest feature importances (raw features)")
plt.tight_layout()
plt.show()

# If RandomForest does signficantly better than NN, that means tree-based model better captures signal.
# If none does much better than majority baseline, likely label noise or features insufficient.

# ---------- Optional: try Polynomial interactions (cheap) ----------
print("\nTrying polynomial interaction features (degree=2 interaction-only) for a LogisticRegression baseline...")
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
Xtr_poly = poly.fit_transform(X_train_s)
Xte_poly = poly.transform(X_test_s)
print("Poly features shape:", Xtr_poly.shape)
lr_poly = LogisticRegression(max_iter=500, multi_class='multinomial', solver='lbfgs')
lr_poly.fit(Xtr_poly, y_train)
y_pred_poly = lr_poly.predict(Xte_poly)
print("LogReg+poly test acc:", accuracy_score(y_test, y_pred_poly))
print(classification_report(y_test, y_pred_poly, digits=4))

# ---------- Improved NN: use class weights and sparse labels ----------
print("\nTraining NN with class weights and sparse loss (this replaces your previous NN training).")

# compute class weights
cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {i: float(w) for i, w in enumerate(cw)}
print("Class weights:", class_weights)

# Build a slightly bigger NN and use sparse_categorical_crossentropy so we can pass integer labels and class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

input_dim = X_train_s.shape[1]
model = Sequential([
    Input(shape=(input_dim,)),
    Dense(128, activation='relu', kernel_regularizer=l2(1e-3)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu', kernel_regularizer=l2(1e-3)),
    BatchNormalization(),
    Dropout(0.15),
    Dense(NUM_TIERS, activation='softmax')
])
model.compile(optimizer=Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

es = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1)
rl = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-6, verbose=1)

history = model.fit(X_train_s, y_train, validation_data=(X_val_s, y_val), epochs=80, batch_size=64,
                    class_weight=class_weights, callbacks=[es, rl], verbose=2)

# Evaluate NN
loss, acc = model.evaluate(X_test_s, y_test, verbose=0)
print(f"NN (class-weighted) test loss: {loss:.4f}, test acc: {acc:.4f}")
y_pred_nn = np.argmax(model.predict(X_test_s), axis=1)
print("NN classification report:")
print(classification_report(y_test, y_pred_nn, digits=4))

# ---------- Summary table ----------
print("\nSUMMARY ACCURACIES:")
print(f"Majority baseline: {majority_acc:.4f}")
print(f"Logistic Regression: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Random Forest:       {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"LogReg + Poly:       {accuracy_score(y_test, y_pred_poly):.4f}")
print(f"NN (class-weighted): {accuracy_score(y_test, y_pred_nn):.4f}")

# ---------- Suggestions after results ----------
print("\nSuggestions:")
print("- If RandomForest outperforms NN, prefer tree-based models for these 8 features.")
print("- If no model improves past ~0.25, inspect label noise / feature signal:")
print("  * Plot per-class feature distributions (boxplots) to see separability.")
print("  * Check CSV labels: are they reliable? Are many images ambiguous between tiers?")
print("  * Consider adding more features (raw seg/object counts, ratios, geometry, image metadata).")
print("- Try SMOTE oversampling (from imblearn.over_sampling import SMOTE) if you see class imbalance helping.")
print("- If you want, I can add a quick SMOTE + RF/NN experiment.")



In [ ]:
# Assume necessary variables (X_train_s, y_train, rf, etc.) are available from the previous run.

# ---------------- 1. FIX: Define ALL 33 Feature Names ----------------
SEG_NAMES = ['SEG_Bicycle','SEG_Bicyclist','SEG_Building','SEG_Bus','SEG_Car','SEG_Curb','SEG_Crosswalk','SEG_FireHydrant','SEG_LaneMarking','SEG_Parking','SEG_Road','SEG_Sidewalk','SEG_StreetLight','SEG_TrafficLight','SEG_TrafficSign','SEG_Truck','SEG_Unlabeled','SEG_Vegetation']
OBJ_NAMES = ['YOLO_Person','YOLO_Bicycle','YOLO_Car','YOLO_Motorcycle','YOLO_Bus','YOLO_Train','YOLO_Truck','YOLO_TrafficLight','YOLO_StopSign','YOLO_Bench','YOLO_Backpack']
RAW_FEATURE_NAMES = SEG_NAMES + OBJ_NAMES

NEW_FEATURE_NAMES = ['Car_Density', 'Ped_Density', 'Bike_Car_Interaction', 'Obstacle_Ratio']
ALL_FEATURE_NAMES = RAW_FEATURE_NAMES + NEW_FEATURE_NAMES
print("Total features:", len(ALL_FEATURE_NAMES))

# ---------------- 2. FIX: Plot Random Forest Feature Importances ----------------
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.metrics import classification_report
from sklearn.utils import class_weight
import numpy as np
import seaborn as sns # Ensure seaborn is available for plotting styles

# Plot RF feature importances (rf was trained in the previous step)
fi = rf.feature_importances_
plt.figure(figsize=(12, 6))
plt.bar(range(len(fi)), fi)
plt.xticks(range(len(fi)), ALL_FEATURE_NAMES, rotation=60, ha='right')
plt.title("RandomForest Feature Importances (33 Features)")
plt.tight_layout()
plt.show()

# ---------------- 3. Train the Improved Neural Network ----------------
print("\nTraining NN with class weights and sparse loss...")

# Compute class weights (to handle minor imbalance and focus on harder classes)
cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {i: float(w) for i, w in enumerate(cw)}
print("Class weights:", class_weights)

# Model Building
NUM_TIERS = 3
input_dim = X_train_s.shape[1]

model = Sequential([
    Input(shape=(input_dim,)),
    Dense(128, activation='relu', kernel_regularizer=l2(1e-3)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu', kernel_regularizer=l2(1e-3)),
    BatchNormalization(),
    Dropout(0.15),
    Dense(NUM_TIERS, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Callbacks
es = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1)
rl = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-6, verbose=1)

# Train
history = model.fit(X_train_s, y_train,
                    validation_data=(X_val_s, y_val),
                    epochs=80,
                    batch_size=64,
                    class_weight=class_weights,
                    callbacks=[es, rl],
                    verbose=2)

# ---------------- 4. Evaluate NN and Final Summary ----------------
loss, acc = model.evaluate(X_test_s, y_test, verbose=0)
y_pred_nn = np.argmax(model.predict(X_test_s, verbose=0), axis=1)

print(f"\nNN (class-weighted) test loss: {loss:.4f}, test acc: {acc:.4f}")
print("NN classification report:")
print(classification_report(y_test, y_pred_nn, digits=4))

# Final Summary (using y_pred_lr and y_pred_poly from previous run)
print("\nSUMMARY ACCURACIES:")
print(f"Majority baseline: 0.3334") # Assuming majority_acc was 0.3334
print(f"Logistic Regression (33 Feat): {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Random Forest (33 Feat):       {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"LogReg + Poly (33 Feat):       {accuracy_score(y_test, y_pred_poly):.4f}")
print(f"NN (class-weighted, 33 Feat):  {acc:.4f}")

In [ ]:
OBJ_CLASSES_TO_KEEP = {
    0: 0,   # Person
    1: 1,   # Bicycle
    2: 2,   # Car
    3: 3,   # Motorcycle
    5: 4,   # Bus
    6: 5,   # Train
    7: 6,   # Truck
    9: 7,   # Traffic Light
    11: 8,  # Stop Sign
    13: 9,  # Bench
    24: 10, # Backpack
}
NEW_MAX_OBJ_LABEL = len(OBJ_CLASSES_TO_KEEP)

SEG_CLASSES_TO_KEEP = {
    1: 0,   # Bicycle
    2: 1,   # Bicyclist
    11: 2,  # Building
    12: 3,  # Bus
    19: 4,  # Car
    24: 5,  # Curb
    26: 6,  # Crosswalk—Plain
    29: 7,  # Fire Hydrant
    32: 8,  # Lane Marking—Crosswalk
    42: 9,  # Parking
    48: 10, # Road
    51: 11, # Sidewalk
    54: 12, # Street Light
    56: 13, # Traffic Light
    58: 14, # Traffic Sign (Front)
    62: 15, # Truck
    64: 16, # Unlabeled
    66: 17, # Vegetation
}
NEW_MAX_SEG_LABEL = len(SEG_CLASSES_TO_KEEP)

# ====================================================================
# 2. UPDATE GLOBAL CONSTANTS
# (Replacing the existing constants)
# ====================================================================
CONTAINER_NAME = "london"
CSV_FILE_NAME = "ai_rank_gpt.csv"
SEGMENTATION_FOLDER = "SemanticSegmentation/json"
DETECTION_FOLDER = "YoloV10/conf_0.7/labels"
DATA_CACHE_FILE = 'feature_data_cache.npz'

SIGMA_THRESHOLD = 4.0

MAX_SEG_LABEL = NEW_MAX_SEG_LABEL
MAX_OBJ_LABEL = NEW_MAX_OBJ_LABEL
NUM_RANK_TIERS = 3


def parse_segmentation(json_data):
    features = np.zeros(MAX_SEG_LABEL)
    try:
        data = json.loads(json_data)
        total_pixels = data.get('height', 1) * data.get('width', 1)
        for item in data.get('label_histogram', []):
            original_idx = item['label_id']
            if original_idx in SEG_CLASSES_TO_KEEP:
                new_idx = SEG_CLASSES_TO_KEEP[original_idx]
                features[new_idx] = item['pixels'] / total_pixels
    except Exception:
        pass
    return features

def parse_yolo(yolo_data):
    features = np.zeros(MAX_OBJ_LABEL)
    try:
        lines = yolo_data.strip().split('\n')
        for line in lines:
            parts = line.split()
            if len(parts) >= 1:
                original_idx = int(parts[0])
                if original_idx in OBJ_CLASSES_TO_KEEP:
                    new_idx = OBJ_CLASSES_TO_KEEP[original_idx]
                    features[new_idx] += 1
    except Exception:
        pass
    return features

def get_blob_service():
    return BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)

def load_data():
    blob_service = get_blob_service()
    container_client = blob_service.get_container_client(CONTAINER_NAME)

    try:
        df = pd.read_csv(CSV_FILE_NAME)
        print(f"✅ Successfully loaded CSV from local path: {CSV_FILE_NAME}")
    except FileNotFoundError:
        print(f"❌ ERROR: The local file '{CSV_FILE_NAME}' was not found in the Colab environment.")
        raise
    except Exception as e:
        print(f"❌ ERROR reading local CSV: {e}")
        raise

    initial_count = len(df)
    df = df[df['ai_sigma'] <= SIGMA_THRESHOLD].copy()
    filtered_count = len(df)
    print(f"🗑️ Filtered {initial_count - filtered_count} image entries where ai_sigma > {SIGMA_THRESHOLD}.")
    print(f"   {filtered_count} image entries remain for feature lookup.")

    if 'rank_ai' not in df.columns:
        raise ValueError("Column 'rank_ai' not found in CSV.")

    df['safety_score'] = pd.qcut(df['rank_ai'], q=NUM_RANK_TIERS, labels=False, duplicates='drop').astype(int) + 1
    df['safety_tier_idx'] = df['safety_score'] - 1

    df['basename'] = df['image_name'].apply(lambda s: os.path.splitext(os.path.basename(str(s)))[0])
    df['root'] = df['basename'].apply(lambda s: s.split('_')[0])

    from collections import defaultdict
    exact_map = defaultdict(list)
    root_map = defaultdict(list)
    for idx, row in df.iterrows():
        exact_map[row['basename']].append(idx)
        root_map[row['root']].append(idx)

    data_map = {}
    for idx, row in df.iterrows():
        data_map[idx] = {
            'target': int(row['safety_tier_idx']),
            'seg_features': np.zeros(MAX_SEG_LABEL),
            'obj_features': np.zeros(MAX_OBJ_LABEL),
            'has_seg': False,
            'has_obj': False
        }

    blobs = list(container_client.list_blobs())
    unmatched_blobs = []
    matched_blob_count = 0

    for blob in tqdm(blobs, desc="Processing Blobs"):
        filename = os.path.basename(blob.name)
        blob_base = os.path.splitext(filename)[0]
        blob_root = blob_base.split('_')[0]

        candidate_idxs = exact_map.get(blob_base, [])

        if not candidate_idxs:
            candidate_idxs = root_map.get(blob_root, [])
            if len(candidate_idxs) != 1:
                candidate_idxs = []

        if not candidate_idxs:
            unmatched_blobs.append(blob.name)
            continue

        for idx in candidate_idxs:
            blob_client = container_client.get_blob_client(blob.name)

            if SEGMENTATION_FOLDER in blob.name and filename.endswith('.json'):
                try:
                    content = blob_client.download_blob().readall().decode('utf-8')
                    data_map[idx]['seg_features'] = parse_segmentation(content)
                    data_map[idx]['has_seg'] = True
                    matched_blob_count += 1
                except UnicodeDecodeError:
                    print(f"Warning: Segmentation JSON file {blob.name} failed UTF-8 decode. Skipping.")
                except Exception as e:
                    # avoid silent failures
                    print(f"Warning: failed to parse segmentation {blob.name}: {e}")

            # detection TXT
            elif DETECTION_FOLDER in blob.name and filename.endswith('.txt'):
                try:
                    content = blob_client.download_blob().readall().decode('utf-8')
                    data_map[idx]['obj_features'] = parse_yolo(content)
                    data_map[idx]['has_obj'] = True
                    matched_blob_count += 1
                except UnicodeDecodeError:
                    print(f"Warning: Detection TXT file {blob.name} failed UTF-8 decode. Skipping.")
                except Exception as e:
                    print(f"Warning: failed to parse detection {blob.name}: {e}")

    seg_only = sum(1 for v in data_map.values() if v['has_seg'] and not v['has_obj'])
    obj_only = sum(1 for v in data_map.values() if v['has_obj'] and not v['has_seg'])
    both = sum(1 for v in data_map.values() if v['has_obj'] and v['has_seg'])
    none = sum(1 for v in data_map.values() if not v['has_obj'] and not v['has_seg'])
    total_rows = len(data_map)

    print(f"\nBlob matching diagnostics:")
    print(f"  Total CSV rows considered: {total_rows}")
    print(f"  Matched blob files processed: {matched_blob_count}")
    print(f"  Rows with segmentation only: {seg_only}")
    print(f"  Rows with detection only:   {obj_only}")
    print(f"  Rows with both features:    {both}")
    print(f"  Rows with NO features:      {none}")

    # Optionally print some unmatched examples for debugging
    if len(unmatched_blobs) > 0:
        print(f"\nSome unmatched blobs (first 10): {unmatched_blobs[:10]}")
        print("If these should match CSV rows, verify the CSV 'image_name' formatting or adjust the root-splitting rule.")

    # Build X,y from rows that have at least one feature
    X = []
    y = []
    for idx, val in data_map.items():
        if val['has_seg'] or val['has_obj']:
            combined_features = np.concatenate((val['seg_features'], val['obj_features']))
            X.append(combined_features)
            y.append(val['target'])

    return np.array(X), np.array(y)



def save_data_cache(X, y, filename=DATA_CACHE_FILE):
    """Saves the combined feature and target arrays to a compressed NPZ file."""
    np.savez_compressed(filename, features=X, targets=y)
    print(f"\n✅ Data cache saved to {filename} (features: {X.shape}, targets: {y.shape})")

def load_data_cache(filename=DATA_CACHE_FILE):
    """Loads the combined feature and target arrays from a compressed NPZ file."""
    try:
        data = np.load(filename)
        X = data['features']
        y = data['targets']
        print(f"\n✅ Data cache loaded from {filename} (features: {X.shape}, targets: {y.shape})")
        return X, y
    except FileNotFoundError:
        print(f"❌ Data cache file {filename} not found. Must run load_data() from Azure.")
        return None, None

X, y = load_data_cache()

if X is None:
    X, y = load_data()
    save_data_cache(X, y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

y_one_hot = to_categorical(y, num_classes=NUM_RANK_TIERS) # <-- CRITICAL CHANGE

X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y_one_hot, test_size=0.1, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=(1/9), random_state=42
)

print(f"Total Samples: {len(X_scaled)}")
print(f"Train Samples (80%): {len(X_train)}")
print(f"Validation Samples (10%): {len(X_val)}")
print(f"Test Samples (10%): {len(X_test)}")

model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.1),
    Dense(NUM_RANK_TIERS, activation='softmax')
])

model.compile(optimizer=Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])



history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
)

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

y_pred_probs = model.predict(X_test[:5])
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_pred_scores = y_pred_classes + 1

print("---")
print(f"Final Test Set Loss: {loss:.4f}")
print(f"Final Test Set Accuracy: {accuracy:.4f}")
print(f"Predictions sample (Safety Scores 1-10): {y_pred_scores}")
# -----------------------------------------------------------------

MODEL_FILE_NAME = 'safety_score_model.keras'
model.save(MODEL_FILE_NAME)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

clf = LogisticRegression(max_iter=2000, multi_class='multinomial')
clf.fit(X_scaled, y)
pred = clf.predict(X_scaled)

print("Training accuracy:", accuracy_score(y, pred))


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

# Get predictions for the entire test set
y_pred_probs_full = model.predict(X_test, verbose=0)
y_pred_classes_full = np.argmax(y_pred_probs_full, axis=1)

# Convert classes (0-9) to Scores (1-10)
y_test_scores = np.argmax(y_test, axis=1) + 1
y_pred_scores = y_pred_classes_full + 1

# Generate and print the confusion matrix array
cm = confusion_matrix(y_test_scores, y_pred_scores)
print("--- Raw Confusion Matrix Array (Scores 1-10) ---")
print(cm)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix

# --- 1. PREPARE CLASSIFICATION DATA ---

# Convert one-hot encoded true labels (y_test) back to integer classes (0-9)
y_test_classes = np.argmax(y_test, axis=1)

# Get predictions for the entire test set
y_pred_probs_full = model.predict(X_test, verbose=0)
y_pred_classes_full = np.argmax(y_pred_probs_full, axis=1)

# Convert classes (0-9) to Scores (1-10) for plotting labels
y_test_scores = y_test_classes + 1
y_pred_scores = y_pred_classes_full + 1

# --- 2. PLOT LOSS AND ACCURACY CURVES ---

plt.figure(figsize=(15, 5))

# Plot 1: Loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss (Categorical Crossentropy)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Plot 2: Accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.savefig('classification_curves.png')
plt.show()
plt.close()

# --- 3. PLOT CONFUSION MATRIX ---

# Generate the confusion matrix
cm = confusion_matrix(y_test_scores, y_pred_scores)
score_labels = [f'{i}' for i in range(1, NUM_RANK_TIERS + 1)]

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=False,
    xticklabels=score_labels,
    yticklabels=score_labels
)
plt.title('Confusion Matrix (Actual vs Predicted Safety Score)')
plt.ylabel('True Safety Score (1-10)')
plt.xlabel('Predicted Safety Score (1-10)')
plt.savefig('confusion_matrix.png')
plt.close()

print("\n✅ Classification plots generated: 'classification_curves.png' and 'confusion_matrix.png'.")